# Vision-based mmWave Beam Prediction in V2X Communications using JEPA

This notebook covers:
1. **Pretraining** — I-JEPA self-supervised pretraining on DeepSense 6G
2. **Beam Prediction** — Frozen I-JEPA encoder downstream evaluation
3. **ResNet18 Baseline** — Supervised beam prediction baseline
4. **Day/Night Classification** — Frozen I-JEPA encoder
5. **LOS/NLOS Classification** — ResNet18 fine-tuned
6. **Few-Shot Adaptation** — I-JEPA vs ResNet18 on e-FLASH dataset

## 1. Setup — Mount Drive and Import Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd
import glob
import matplotlib.pyplot as plt
import os, sys, random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import shutil
from collections import Counter, defaultdict
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print('Libraries loaded.')

## 2. Pretraining — I-JEPA on DeepSense 6G

> **Skip this section if the pretrained checkpoint already exists.**

In [ ]:
# Copy images from scenario folders into a unified IJEPA_Dataset folder
# Only run once — skip if IJEPA_Dataset already exists

base = '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07'
sources = [
    os.path.join(base, 'Final_Dataset_Train'),
    os.path.join(base, 'Final_Dataset_Train', 'New Scenarios'),
]
output_root = os.path.join(base, 'IJEPA_Dataset')
os.makedirs(output_root, exist_ok=True)

total = 0
for input_root in sources:
    for scenario_folder in os.listdir(input_root):
        input_scenario = os.path.join(input_root, scenario_folder)
        if not os.path.isdir(input_scenario):
            continue
        output_scenario = os.path.join(output_root, scenario_folder)
        os.makedirs(output_scenario, exist_ok=True)
        images = [f for f in os.listdir(input_scenario) if f.endswith('.jpg')]
        for fname in images:
            shutil.copy2(os.path.join(input_scenario, fname), os.path.join(output_scenario, fname))
            total += 1
        print(f'Copied {scenario_folder}: {len(images)} images')

print(f'\nTotal images copied: {total}')

In [ ]:
# Create checkpoint directory and run pretraining
os.makedirs(os.path.join(base, 'ijepa_checkpoints'), exist_ok=True)

%cd '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07/ijepa_modified_k'
!python main.py --fname configs/deepsense6G_config.yaml --devices cuda:0

## 3. Beam Prediction — Frozen I-JEPA Encoder

In [ ]:
sys.path.append('/content/drive/MyDrive/Colab Notebooks/ENEL 619 07/ijepa_modified_k')
from src.models.vision_transformer import vit_base

base_dir        = '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07'
csv_root        = os.path.join(base_dir, 'Final Project dataset')
image_root      = os.path.join(base_dir, 'Final_Dataset_Train')
checkpoint_path = os.path.join(base_dir, 'ijepa_checkpoints/deepsense6G_hpc-latest.pth.tar')
results_dir     = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)

device     = 'cuda' if torch.cuda.is_available() else 'cpu'
scenarios  = [9]
beam_cols  = ['unit1_beam_index', 'unit1_beam', 'best_beam']
top_k      = [1, 2, 3, 5]
batch_size = 64
num_epochs = 100
lr         = 1e-3
torch.manual_seed(42)
print(f'Device: {device}')

In [ ]:
def load_data():
    records = []
    for s in scenarios:
        csv_path = os.path.join(csv_root, f'scenario{s}', f'scenario{s}.csv')
        img_dir  = os.path.join(image_root, f'camera_data_s{s}')
        if not os.path.exists(csv_path) or not os.path.exists(img_dir):
            print(f'[SKIP] Scenario {s}')
            continue
        df       = pd.read_csv(csv_path)
        beam_col = next((c for c in beam_cols if c in df.columns), None)
        if beam_col is None or 'unit1_rgb' not in df.columns:
            print(f'[SKIP] Scenario {s} - missing columns')
            continue
        df = df.dropna(subset=[beam_col, 'unit1_rgb'])
        df['beam_label'] = df[beam_col].astype(int)
        df['img_path']   = df['unit1_rgb'].apply(lambda p: os.path.join(img_dir, os.path.basename(str(p))))
        df['scenario']   = s
        df = df[df['img_path'].apply(os.path.exists)]
        print(f'Scenario {s}: {len(df)} samples')
        records.append(df[['img_path', 'beam_label', 'scenario']])
    return pd.concat(records, ignore_index=True)

def split(df):
    trains, tests = [], []
    for s in df['scenario'].unique():
        sub = df[df['scenario'] == s]
        n   = int(len(sub) * 0.7)
        trains.append(sub.iloc[:n])
        tests.append(sub.iloc[n:])
    return pd.concat(trains), pd.concat(tests)

class BeamDataset(Dataset):
    def __init__(self, df, beam_to_idx, transform):
        self.df, self.beam_to_idx, self.transform = df.reset_index(drop=True), beam_to_idx, transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['img_path']).convert('RGB')
        return self.transform(img), self.beam_to_idx[row['beam_label']]

def load_encoder():
    encoder = vit_base(patch_size=16)
    ckpt    = torch.load(checkpoint_path, map_location='cpu')
    sd      = ckpt.get('target_encoder', ckpt.get('encoder', ckpt))
    sd      = {k.replace('module.', ''): v for k, v in sd.items()}
    encoder.load_state_dict(sd, strict=False)
    for p in encoder.parameters():
        p.requires_grad = False
    encoder.eval()
    return encoder

@torch.no_grad()
def extract(encoder, loader):
    encoder.to(device)
    feats, labels = [], []
    for imgs, lbs in loader:
        out = encoder(imgs.to(device))
        feats.append((out.mean(dim=1) if out.dim() == 3 else out).cpu())
        labels.append(lbs)
    return torch.cat(feats), torch.cat(labels)

class Head(nn.Module):
    def __init__(self, in_dim, n):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, 512),
                                 nn.GELU(), nn.Dropout(0.2), nn.Linear(512, n))
    def forward(self, x): return self.net(x)

def train_head(feats, labels, n):
    head      = Head(feats.shape[1], n).to(device)
    w         = labels.bincount(minlength=n).float().clamp(min=1)
    w         = (w.sum() / (n * w)).to(device)
    opt       = torch.optim.Adam(head.parameters(), lr=lr)
    sched     = torch.optim.lr_scheduler.CosineAnnealingLR(opt, num_epochs)
    criterion = nn.CrossEntropyLoss(weight=w)
    loader    = DataLoader(torch.utils.data.TensorDataset(feats, labels), batch_size=256, shuffle=True)
    for epoch in range(1, num_epochs + 1):
        head.train()
        for xb, yb in loader:
            opt.zero_grad()
            criterion(head(xb.to(device)), yb.to(device)).backward()
            opt.step()
        sched.step()
        if epoch % 20 == 0 or epoch == num_epochs:
            print(f'Epoch {epoch}/{num_epochs}')
    head.eval()
    return head

def topk_acc(logits, labels):
    return {k: 100. * logits.topk(min(k, logits.shape[1]), dim=1).indices
                        .eq(labels.unsqueeze(1)).any(1).float().mean().item() for k in top_k}

def plot_topk(accs, title):
    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar([f'Top-{k}' for k in top_k], [accs[k] for k in top_k],
                  color=['#2196F3', '#4CAF50', '#FF9800', '#F44336'], width=0.5)
    for bar, k in zip(bars, top_k):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                f'{accs[k]:.2f}%', ha='center', fontsize=11)
    ax.set_ylim(0, 110)
    ax.set_xlabel('Top-k')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'topk_ijepa.png'), dpi=150)
    plt.show()

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

df          = load_data()
train_df, test_df = split(df)
all_beams   = sorted(df['beam_label'].unique())
beam_to_idx = {b: i for i, b in enumerate(all_beams)}
n_classes   = len(all_beams)
print(f'Train: {len(train_df)} | Test: {len(test_df)} | Classes: {n_classes}')

train_loader = DataLoader(BeamDataset(train_df, beam_to_idx, transform), batch_size=batch_size, shuffle=False, num_workers=2)
test_loader  = DataLoader(BeamDataset(test_df,  beam_to_idx, transform), batch_size=batch_size, shuffle=False, num_workers=2)

print('\nLoading encoder...')
encoder = load_encoder()

print('Extracting features...')
train_feats, train_labels = extract(encoder, train_loader)
test_feats,  test_labels  = extract(encoder, test_loader)
print(f'Feature shape: {train_feats.shape}')

print('\nTraining MLP head...')
head = train_head(train_feats, train_labels, n_classes)

with torch.no_grad():
    logits = head(test_feats.to(device)).cpu()

accs = topk_acc(logits, test_labels)
print('\n--- I-JEPA Beam Prediction Results ---')
for k in top_k:
    print(f'  Top-{k}: {accs[k]:.2f}%')

plot_topk(accs, f'Beam Prediction Top-k — I-JEPA Frozen\nScenarios {scenarios}')

## 4. Beam Prediction — ResNet18 Supervised Baseline

In [ ]:
scenarios  = [5]
epochs     = 50
patience   = 5
lr         = 1e-3
weight_decay = 0.01
seed       = 42

def load_data_resnet():
    all_dfs = []
    for s in scenarios:
        csv_files = glob.glob(os.path.join(csv_root, f'scenario{s}', '*.csv'))
        if not csv_files:
            continue
        df       = pd.read_csv(csv_files[0])
        beam_col = next((c for c in ['best_beam', 'unit1_beam_index', 'unit1_beam'] if c in df.columns), None)
        img_col  = next((c for c in ['unit1_rgb', 'unit1_camera', 'image_path'] if c in df.columns), None)
        if beam_col is None or img_col is None:
            continue
        img_folder = os.path.join(image_root, f'camera_data_s{s}')
        if not os.path.exists(img_folder):
            continue
        available = set(os.listdir(img_folder))
        sdf = pd.DataFrame({
            'image_filename': df[img_col].apply(lambda x: os.path.basename(str(x))),
            'beam_index': df[beam_col].astype(int),
            'scenario': s
        })
        sdf = sdf[sdf['image_filename'].isin(available)].reset_index(drop=True)
        all_dfs.append(sdf)
        print(f'Scenario {s}: {len(sdf)} samples')
    return pd.concat(all_dfs, ignore_index=True)

def prepare_labels(df):
    unique_beams  = sorted(df['beam_index'].unique())
    beam_to_label = {b: i for i, b in enumerate(unique_beams)}
    df['label']   = df['beam_index'].map(beam_to_label)
    return df, len(unique_beams)

def split_data(df):
    n       = len(df)
    indices = np.arange(n)
    np.random.RandomState(seed).shuffle(indices)
    train_end = int(n * 0.70)
    val_end   = int(n * 0.85)
    return (df.iloc[indices[:train_end]].reset_index(drop=True),
            df.iloc[indices[train_end:val_end]].reset_index(drop=True),
            df.iloc[indices[val_end:]].reset_index(drop=True))

class BeamDatasetResNet(Dataset):
    def __init__(self, dataframe, image_root, transform):
        self.df         = dataframe.reset_index(drop=True)
        self.image_root = image_root
        self.transform  = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        try:
            row      = self.df.iloc[idx]
            img_path = os.path.join(self.image_root, f'camera_data_s{row["scenario"]}', row['image_filename'])
            return self.transform(Image.open(img_path).convert('RGB')), row['label']
        except Exception:
            return self.__getitem__((idx + 1) % len(self.df))

def topk_accuracy(logits, labels, topk=(1, 2, 3, 5)):
    with torch.no_grad():
        maxk  = max(topk)
        _, pred = logits.topk(maxk, 1, True, True)
        correct = pred.t().eq(labels.view(1, -1).expand_as(pred.t()))
        return {f'top{k}': correct[:k].reshape(-1).float().sum().item() / labels.size(0) * 100 for k in topk}

def train_resnet(model, train_loader, val_loader):
    criterion   = nn.CrossEntropyLoss()
    optimizer   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_top3   = 0
    patience_ctr = 0
    for epoch in range(1, epochs + 1):
        model.train()
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), lbls)
            loss.backward()
            optimizer.step()
        model.eval()
        all_logits, all_labels = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                all_logits.append(model(imgs.to(device)).cpu())
                all_labels.append(lbls)
        val_res = topk_accuracy(torch.cat(all_logits), torch.cat(all_labels))
        if epoch % 10 == 0:
            print(f'Epoch {epoch}/{epochs} | Val Top-3: {val_res["top3"]:.2f}%')
        if val_res['top3'] > best_top3:
            best_top3   = val_res['top3']
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'Early stopping at epoch {epoch}')
                break
    return best_state

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

eval_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

df_r, num_classes = prepare_labels(load_data_resnet())
train_df_r, val_df_r, test_df_r = split_data(df_r)
print(f'Train: {len(train_df_r)} | Val: {len(val_df_r)} | Test: {len(test_df_r)} | Classes: {num_classes}')

train_loader_r = DataLoader(BeamDatasetResNet(train_df_r, image_root, train_transform), batch_size=64, shuffle=True,  num_workers=2)
val_loader_r   = DataLoader(BeamDatasetResNet(val_df_r,   image_root, eval_transform),  batch_size=64, shuffle=False, num_workers=2)
test_loader_r  = DataLoader(BeamDatasetResNet(test_df_r,  image_root, eval_transform),  batch_size=64, shuffle=False, num_workers=2)

model    = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model    = model.to(device)

best_state = train_resnet(model, train_loader_r, val_loader_r)
model.load_state_dict(best_state)
model.eval()

all_logits, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in test_loader_r:
        all_logits.append(model(imgs.to(device)).cpu())
        all_labels.append(lbls)
test_res = topk_accuracy(torch.cat(all_logits), torch.cat(all_labels))

print('\n--- ResNet18 Beam Prediction Results ---')
for k in [1, 2, 3, 5]:
    print(f'  Top-{k}: {test_res[f"top{k}"]:.2f}%')

fig, ax = plt.subplots(figsize=(6, 4))
vals = [test_res[f'top{k}'] for k in [1, 2, 3, 5]]
ax.bar(['Top-1', 'Top-2', 'Top-3', 'Top-5'], vals, width=0.4, color='steelblue')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Beam Prediction — ResNet18 Baseline')
for i, v in enumerate(vals):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'resnet18_baseline.png'), dpi=150)
plt.show()

## 5. Day/Night Classification — Frozen I-JEPA Encoder

In [ ]:
day_scenarios   = [1, 3, 6, 7, 8, 9]
night_scenarios = [2, 4, 14, 33, 34]
num_epochs_dn   = 100
lr_dn           = 1e-3

def load_daynight():
    records = []
    for label, scenario_list in [(0, day_scenarios), (1, night_scenarios)]:
        tag = 'Day' if label == 0 else 'Night'
        for s in scenario_list:
            for subfolder in ['', 'New Scenarios']:
                csv_path = os.path.join(csv_root, subfolder, f'scenario{s}', f'scenario{s}.csv')
                if os.path.exists(csv_path):
                    break
            else:
                print(f'[SKIP] Scenario {s} ({tag})')
                continue
            df = pd.read_csv(csv_path).dropna(subset=['unit1_rgb'])
            df['img_path'] = df['unit1_rgb'].apply(
                lambda p: os.path.join(image_root, f'camera_data_s{s}', os.path.basename(str(p))))
            df['label']    = label
            df['scenario'] = s
            sub = df[['img_path', 'label', 'scenario']]
            sub = sub[sub['img_path'].apply(os.path.exists)]
            records.append(sub)
            print(f'Scenario {s} ({tag}): {len(sub)} samples')
    return pd.concat(records, ignore_index=True)

class DNDataset(Dataset):
    def __init__(self, df, transform):
        self.df, self.transform = df.reset_index(drop=True), transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.transform(Image.open(row['img_path']).convert('RGB')), int(row['label'])

class LinearHead(nn.Module):
    def __init__(self, in_dim=768, n=2):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, n))
    def forward(self, x): return self.net(x)

dn_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

df_dn = load_daynight()
trains, tests = [], []
for s in df_dn['scenario'].unique():
    sub = df_dn[df_dn['scenario'] == s]
    n   = int(len(sub) * 0.7)
    trains.append(sub.iloc[:n]); tests.append(sub.iloc[n:])
train_dn, test_dn = pd.concat(trains), pd.concat(tests)
print(f'Train: {len(train_dn)} | Test: {len(test_dn)}')

dn_train_loader = DataLoader(DNDataset(train_dn, dn_transform), batch_size=64, shuffle=False, num_workers=2)
dn_test_loader  = DataLoader(DNDataset(test_dn,  dn_transform), batch_size=64, shuffle=False, num_workers=2)

dn_train_feats, dn_train_labels = extract(encoder, dn_train_loader)
dn_test_feats,  dn_test_labels  = extract(encoder, dn_test_loader)

dn_head   = LinearHead().to(device)
dn_counts = torch.bincount(dn_train_labels, minlength=2).float()
dn_w      = (dn_counts.sum() / (2 * dn_counts.clamp(min=1))).to(device)
dn_opt    = torch.optim.Adam(dn_head.parameters(), lr=lr_dn)
dn_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(dn_opt, num_epochs_dn)
dn_crit   = nn.CrossEntropyLoss(weight=dn_w)
dn_loader = DataLoader(torch.utils.data.TensorDataset(dn_train_feats, dn_train_labels), batch_size=256, shuffle=True)

for epoch in range(1, num_epochs_dn + 1):
    dn_head.train()
    for xb, yb in dn_loader:
        dn_opt.zero_grad()
        dn_crit(dn_head(xb.to(device)), yb.to(device)).backward()
        dn_opt.step()
    dn_sched.step()
    if epoch % 20 == 0 or epoch == num_epochs_dn:
        print(f'Epoch {epoch}/{num_epochs_dn}')

dn_head.eval()
with torch.no_grad():
    dn_logits = dn_head(dn_test_feats.to(device)).cpu()
dn_preds  = dn_logits.argmax(1).numpy()
dn_labels = dn_test_labels.numpy()

acc       = 100. * (dn_preds == dn_labels).mean()
day_acc   = 100. * (dn_preds[dn_labels==0] == 0).mean()
night_acc = 100. * (dn_preds[dn_labels==1] == 1).mean()
print(f'\nOverall: {acc:.2f}%  |  Day: {day_acc:.2f}%  |  Night: {night_acc:.2f}%')

from sklearn.metrics import confusion_matrix
cm      = confusion_matrix(dn_labels, dn_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
names   = ['Day', 'Night']
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(names); ax.set_yticklabels(names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Day / Night — I-JEPA Frozen Encoder')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center',
                color='white' if cm_norm[i, j] > 0.5 else 'black', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'daynight_ijepa.png'), dpi=150)
plt.show()

## 6. Few-Shot Beam Prediction — I-JEPA vs ResNet18 on e-FLASH

In [ ]:
eflash_base  = os.path.join(base_dir, 'E-flash dataset', 'Cat2', 'ped_left_to_right', 'episode_3', 'npz')
shot_counts  = [10, 20, 30]
top_k_fs     = [1, 2, 3, 5]
epochs_fs    = 200
lr_fs        = 1e-3
seed_fs      = 42

IMAGE_KEYS = ['data', 'images', 'image', 'frames', 'rgb', 'arr_0']
BEAM_KEYS  = ['beams', 'beam_index', 'best_beam', 'label', 'beam', 'arr_0', 'snr']

import numpy as np

def inspect_and_load_npz():
    img_d  = np.load(os.path.join(eflash_base, 'image4.npz'), allow_pickle=True)
    rf_d   = np.load(os.path.join(eflash_base, 'rf.npz'),     allow_pickle=True)
    print('image4 keys:', img_d.files)
    print('rf keys:    ', rf_d.files)
    img_key  = next((k for k in IMAGE_KEYS if k in img_d.files), img_d.files[0])
    beam_key = next((k for k in BEAM_KEYS  if k in rf_d.files),  rf_d.files[0])
    print(f'Using image key: "{img_key}"  |  beam key: "{beam_key}"')
    images = img_d[img_key]
    beams  = rf_d[beam_key]
    if beams.ndim == 2:
        beams = beams.argmax(axis=1)
    beams = beams.astype(int).flatten()
    if images.ndim == 4 and images.shape[1] == 3:
        images = images.transpose(0, 2, 3, 1)
    print(f'Images: {images.shape} | Beams: {beams.shape} | Classes: {len(np.unique(beams))}')
    return images, beams

images_fs, beams_fs = inspect_and_load_npz()

unique_beams_fs = sorted(np.unique(beams_fs).tolist())
b2i_fs          = {b: i for i, b in enumerate(unique_beams_fs)}
num_classes_fs  = len(unique_beams_fs)
labels_idx_fs   = np.array([b2i_fs[b] for b in beams_fs])

class FrameDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images, self.labels, self.transform = images, labels, transform
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx].astype(np.uint8)).convert('RGB')
        return self.transform(img), int(self.labels[idx])

fs_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

full_ds     = FrameDataset(images_fs, labels_idx_fs, fs_transform)
full_loader = DataLoader(full_ds, batch_size=64, shuffle=False, num_workers=2)

def load_ijepa_encoder():
    enc = vit_base(patch_size=16)
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    sd   = ckpt.get('target_encoder', ckpt.get('encoder', ckpt))
    sd   = {k.replace('module.', ''): v for k, v in sd.items()}
    enc.load_state_dict(sd, strict=False)
    for p in enc.parameters():
        p.requires_grad = False
    enc.eval()
    return enc

def load_resnet18():
    resnet  = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    encoder = nn.Sequential(*list(resnet.children())[:-1])
    for p in encoder.parameters():
        p.requires_grad = False
    encoder.eval()
    return encoder

ijepa_encoder = load_ijepa_encoder().to(device)
resnet_model  = load_resnet18().to(device)

def extract_features(encoder, images, labels, encoder_type):
    dataset = FrameDataset(images, labels, fs_transform)
    loader  = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)
    all_feats, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            if encoder_type == 'ijepa':
                tokens = encoder(imgs.to(device))
                feats  = tokens.mean(dim=1) if tokens.dim() == 3 else tokens
            else:
                feats = encoder(imgs.to(device)).squeeze(-1).squeeze(-1)
            all_feats.append(feats.cpu())
            all_labels.append(lbls)
    return torch.cat(all_feats), torch.cat(all_labels)

ijepa_feats,  all_labels = extract_features(ijepa_encoder, images_fs, labels_idx_fs, 'ijepa')
resnet_feats, _          = extract_features(resnet_model,  images_fs, labels_idx_fs, 'resnet')

def sample_fewshot(feats, labels, n_shots, seed):
    rng = np.random.RandomState(seed)
    support_idx, query_idx = [], []
    for cls in range(num_classes_fs):
        idx  = (labels == cls).nonzero(as_tuple=True)[0].tolist()
        if len(idx) == 0:
            continue
        n      = min(n_shots, len(idx))
        chosen = rng.choice(idx, n, replace=False).tolist()
        support_idx.extend(chosen)
        query_idx.extend([i for i in idx if i not in chosen])
    return feats[support_idx], labels[support_idx], feats[query_idx], labels[query_idx]

class LinearHead_FS(nn.Module):
    def __init__(self, in_dim, n):
        super().__init__()
        self.fc = nn.Linear(in_dim, n)
    def forward(self, x): return self.fc(x)

def train_head_fs(support_feats, support_labels, in_dim):
    head      = LinearHead_FS(in_dim, num_classes_fs).to(device)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr_fs, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    dataset   = torch.utils.data.TensorDataset(support_feats, support_labels)
    loader    = DataLoader(dataset, batch_size=min(32, len(dataset)), shuffle=True)
    head.train()
    for _ in range(epochs_fs):
        for xb, yb in loader:
            optimizer.zero_grad()
            criterion(head(xb.to(device)), yb.to(device)).backward()
            optimizer.step()
    head.eval()
    return head

def topk_accuracy_fs(logits, labels):
    results = {}
    for k in top_k_fs:
        k_actual    = min(k, logits.shape[1])
        topk_preds  = logits.topk(k_actual, dim=1).indices
        correct     = topk_preds.eq(labels.unsqueeze(1)).any(1).float().mean().item()
        results[f'top{k}'] = 100.0 * correct
    return results

ijepa_results, resnet_results = {}, {}
for n_shots in shot_counts:
    sup_i, sup_l, qry_i, qry_l = sample_fewshot(ijepa_feats,  all_labels, n_shots, seed_fs)
    sup_r, _,     qry_r, _     = sample_fewshot(resnet_feats, all_labels, n_shots, seed_fs)

    ijepa_head  = train_head_fs(sup_i, sup_l, ijepa_feats.shape[1])
    resnet_head = train_head_fs(sup_r, sup_l, resnet_feats.shape[1])

    with torch.no_grad():
        ijepa_results[n_shots]  = topk_accuracy_fs(ijepa_head(qry_i.to(device)).cpu(),  qry_l)
        resnet_results[n_shots] = topk_accuracy_fs(resnet_head(qry_r.to(device)).cpu(), qry_l)

    print(f'{n_shots}-shot | I-JEPA Top-1: {ijepa_results[n_shots]["top1"]:.1f}% | ResNet18 Top-1: {resnet_results[n_shots]["top1"]:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
n_groups = len(shot_counts)
width    = 0.18
colors   = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
x        = np.arange(n_groups)

for ax_idx, (results, title) in enumerate([(ijepa_results, 'I-JEPA'), (resnet_results, 'ResNet-18')]):
    ax = axes[ax_idx]
    for i, k in enumerate(top_k_fs):
        values = [results[s][f'top{k}'] for s in shot_counts]
        offset = (i - 2 + 0.5) * width
        bars   = ax.bar(x + offset, values, width, label=f'Top-{k}', color=colors[i])
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{val:.1f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{s}-shot' for s in shot_counts])
    ax.set_xlabel('Shots per Class')
    ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 110)
    ax.set_title(title)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Few-Shot Beam Prediction — e-FLASH Cat2 Episode 3', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'fewshot_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()